# Interactive flagging of timeseries data

## Approach and usage
Data are loaded from a .parquet file into a polars dataframe. 
Two auxiliary columns are added, namely '\_flag_' and '\_color_'.
Date are then plotted on an interactive canvas.
User can press and hold a key to determine the flag (currently, 1-3 and Esc) and pick points with the mouse.
Alternatively, user can activate the zoom tool, select an area, and flag all visible points at once by pressing a key.
User can save the data with the flags by clicking on the save button. This will rename the flag column to 'f_variable', drop the color column, and save the data as .parquet file under the level2 branch.
User can switch between variables seamlessly.

Data from level2 folder can be re-loaded and flagging continued.

author: joerg.klausen@meteoswiss.ch

[TODO] re-arrange widgets, make info box wider

In [ ]:
import os
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.dates
import numpy as np
import polars as pl

from ipyfilechooser import FileChooser
from ipywidgets.widgets import Dropdown
from ipywidgets import Button, Text
from ipywidgets import VBox, HBox
from ipywidgets import Output
from IPython.display import display

# activate the matplotlib widget for Jupyter
%matplotlib ipympl

# file related configurations
root_dir = os.path.join(os.getcwd(), "data")
source_dir = "level1"
target_dir = "level2"

# dataframe column label configurations
dtm = "dtm"
flags = "_flag_"
colors = "_color_"

# other configurations
keys = {"escape": {"flag": None, "color": "magenta", "meaning": "unflagged"},
        "0": {"flag": 0, "color": "blue", "meaning": "valid"},
        "1": {"flag": 1, "color": "red", "meaning": "invalid"},
        "2": {"flag": 2, "color": "gray", "meaning": "uncertain"},
        "3": {"flag": 3, "color": "cyan", "meaning": "zero check"},
        "4": {"flag": 4, "color": "brown", "meaning": "span check"},}
flag_col_prefix = "f_"
new_file_on_save = False

# select file
file_chooser = FileChooser(select_desc="Select file", path=os.path.join(root_dir, source_dir), filter_pattern="*.parquet")
display(file_chooser)

In [ ]:
# read data into dataframe
df = pl.read_parquet(source=file_chooser.selected)
df.top_k(k=5, by=dtm)

In [ ]:
def on_dropdown_value_selected(change):
    global df, sc, variable
    old = change.old
    variable = change.new
    infobox.value = f"1. {old} > {variable}"
    if variable and (variable != old):
        infobox.value = f"2. {old} > {variable}"
        f_variable = f"{flag_col_prefix}{variable}"
        
        df = df.with_columns(pl.lit(keys["escape"]["color"], dtype=pl.Utf8).alias(colors))
        
        # load flags if they exist already and update colors
        if flags in df.columns:
            # <flags> column already exists, but user has switched to another variable > keep previous flags as column <{flag_col_prefix}{old}>
            infobox.value = f"3. {old} > {variable}"
            df = df.rename({flags: f"{flag_col_prefix}{old}"})
        if f_variable in df.columns:
            # flag column already exists for the selected variable from earlier flagging > rename to <flags> to continue flagging
            infobox.value = f"4. {old} > {variable}"
            df = df.rename({f_variable: flags})
            # set colors according to flags
            if colors in df.columns:
                infobox.value = f"5. {old} > {variable}"
                for k in keys.keys():
                    if keys[k]["flag"] is not None:
                        df = df.with_columns(pl.when(pl.col(flags) == keys[k]["flag"])
                                            .then(pl.lit(keys[k]["color"]))
                                            .otherwise(pl.col(colors)).alias(colors))             
            else:
                infobox.value = f"6. {old} > {variable}"
                # variable has been flagged before, but is newly selected > set up colors according to existing flags
        else:
            # user has chosen a variable that has never been flagged > initialize <flags> and <colors> columns
            infobox.value = f"7. {old} > {variable}"
            df = df.with_columns(pl.lit(keys["escape"]["flag"], dtype=pl.Int8).alias(flags))
            # df = df.with_columns(pl.lit(keys["escape"]["color"], dtype=pl.Utf8).alias(colors))    
        
        ax.cla()
        ax.set_title('ezFlag - Interactive data flagging')
        sc = ax.scatter(df[dtm], df[variable], c=df[colors], alpha=0.7, s=10, picker=5)
        # fig.canvas.draw_idle()


def on_picked_flag_point(event):
    """
    event.mouseevent.key : None, Any character, shift, control, win (cf. https://matplotlib.org/stable/users/explain/figure/event_handling.html#event-attributes)
    event.mouseevent.button : 1: left, 2: middle, 3: right
    event.ind : index of point picked. NB: the index is set when the figure is created for the first time, so is unaffected by zooming.
    """
    global df

    infobox.value = f"Zoom OFF & key = '{event.mouseevent.key}'. Point with index = {event.ind} selected."
    if ax.get_navigate_mode() is None:
        if keys.get(event.mouseevent.key):
            flag = keys[event.mouseevent.key]["flag"]
            color = keys[event.mouseevent.key]["color"]
            df[event.ind, flags] = flag
            df[event.ind, colors] = color
            sc.set_color(df[colors])
            fig.canvas.draw_idle()
        else:
            infobox.value = f"Zoom OFF & point picked, but key '{event.mouseevent.key}' not assigned." 


def on_key_pressed_flag_points(event):
    global df, variable, condition

    infobox.value = f"Zoom ON & key = '{event.key}' pressed."
    if ax.get_navigate_mode() == "ZOOM":
        if keys.get(event.key):
            flag = keys[event.key]["flag"]
            color = keys[event.key]["color"]
            infobox.value = f"flag = {flag}, color = {color}"
            zoom_xlim = ax.get_xlim()
            zoom_xlim = [matplotlib.dates.num2date(x, tz=None).replace(tzinfo=None) for x in zoom_xlim]
            zoom_ylim = ax.get_ylim()
            condition = (pl.col(dtm) > zoom_xlim[0]) & (pl.col(dtm) < zoom_xlim[1]) & (pl.col(variable) > zoom_ylim[0]) & (pl.col(variable) < zoom_ylim[1])
            df = df.with_columns([pl.when(condition)
                                .then(pl.lit(color))
                                .otherwise(pl.col(colors)).alias(colors),
                                pl.when(condition)
                                .then(pl.lit(flag))
                                .otherwise(pl.col(flags)).alias(flags),])
            sc.set_color(df[colors])
            fig.canvas.draw_idle()
        else:
            infobox.value = f"Zoom ON, but key '{event.key}' not assigned."


def on_clicked_save_data(event):
    global df

    # rename flag column, drop color column
    df = df.rename({flags: f"{flag_col_prefix}{variable}"})
    df = df.drop(colors)

    # set file name for level2 data file and save file
    target_file = os.path.join(file_chooser.selected_path.replace(source_dir, target_dir), file_chooser.selected_filename)
    if new_file_on_save:
        if os.path.exists(target_file):
            infobox.value = f"'{os.path.basename(target_file)}' exists already. A unique name will be created."
            target_file += f"-{datetime.now().strftime('%Y%m%d%H%M%S')}"
    os.makedirs(os.path.dirname(target_file), exist_ok=True)
    df.write_parquet(target_file)
    ax.cla()
    dropdown_variable_select.value = None


# prepare figure and create widget for display
fig = plt.figure(figsize=(10, 5))
ax = fig.subplots()

# create input widgets
dropdown_variable_select = Dropdown(
    value=None, 
    options=df.columns, 
    description="Select variable")
button_save_data = Button(description="Save data")

# create output widgets
figure = Output()
infobox = Text()

# group widgets horizontally, then vertically
hbox = HBox([button_save_data, infobox])
layout = VBox([dropdown_variable_select, figure, hbox]) 

# Display widgets
display(layout)

# # connect events
button_save_data.on_click(on_clicked_save_data)
dropdown_variable_select.observe(on_dropdown_value_selected, names='value')
cid_pick = fig.canvas.mpl_connect('pick_event', on_picked_flag_point)
cid_key_press = fig.canvas.mpl_connect('key_press_event', on_key_pressed_flag_points)

# display the plot
plt.show()

In [ ]:
df.columns

In [ ]:
# # interactive data flagging demo
#
# import matplotlib.pyplot as plt
# import numpy as np
# from ipywidgets import Button, Text
# from ipywidgets import VBox, HBox
# from IPython.display import display
# from ipywidgets import Output
# import polars as pl

# # Activate the matplotlib widget for Jupyter
# %matplotlib ipympl

# # Generate some random data
# np.random.seed(0)
# x = np.random.randn(100)
# y = np.random.randn(100)

# df = pl.DataFrame({"x": x, "y": y, "flag": [None for i in range(100)], "color": ["blue" for i in range(100)]})
# df = df.with_columns(pl.col("flag").cast(pl.Int8).alias("flag"))

# # configure meaning of keys
# keys = {"1": {"flag": 1, "color": "red"},
#        "2": {"flag": 2, "color": "gray"},
#        "3": {"flag": 3, "color": "cyan"},
#        "escape": {"flag": None, "color": "blue"}}

# fig, ax = plt.subplots()
# sc = ax.scatter(df["x"], df["y"], c=df["color"], s=10, picker=10, visible=True)


# def on_picked_flag_point(event):
#     """
#     event.mouseevent.key : None, Any character, shift, control, win (cf. https://matplotlib.org/stable/users/explain/figure/event_handling.html#event-attributes)
#     event.mouseevent.button : 1: left, 2: middle, 3: right
#     event.ind : index of point picked. NB: the index is set when the figure is created for the first time, so is unaffected by zooming.
#     """
#     global df

#     text_display.value = f"Zoom OFF & key = '{event.mouseevent.key}'. Point with index = {event.ind} selected."
#     if ax.get_navigate_mode() is None:
#         if keys.get(event.mouseevent.key):
#             flag = keys[event.mouseevent.key]["flag"]
#             color = keys[event.mouseevent.key]["color"]
#             df[event.ind, "flag"] = flag
#             df[event.ind, "color"] = color
#             sc.set_color(df["color"])
#             fig.canvas.draw_idle()
#         else:
#             text_display.value = f"Zoom OFF & point picked, but key '{event.mouseevent.key}' not assigned." 


# def on_key_pressed_flag_points(event):
#     global df

#     text_display.value = f"Zoom ON & key = '{event.key}' pressed."
#     if ax.get_navigate_mode() == "ZOOM":
#         if keys.get(event.key):
#             flag = keys[event.key]["flag"]
#             color = keys[event.key]["color"]
#             zoom_xlim = ax.get_xlim()
#             zoom_ylim = ax.get_ylim()
#             condition = (pl.col("x") > zoom_xlim[0]) & (pl.col("x") < zoom_xlim[1]) & (pl.col("y") > zoom_ylim[0]) & (pl.col("y") < zoom_ylim[1])
#             df = df.with_columns([pl.when(condition)
#                                 .then(pl.lit(color))
#                                 .otherwise(pl.col("color")).alias("color"),
#                                 pl.when(condition)
#                                 .then(pl.lit(flag))
#                                 .otherwise(pl.col("flag")).alias("flag"),])
#             sc.set_color(df["color"])
#             fig.canvas.draw_idle()
#         else:
#             text_display.value = f"Zoom ON, but key '{event.key}' not assigned."


# def on_clicked_save_data(event):
#     global df
#     text_display.value = "on_clicked_save_data: not yet implemented."
#     print(df.schema)


# button_save_data = Button(description="Save data")
# button_save_data.on_click(on_clicked_save_data)

# # create text display
# text_display = Text()

# # arrange buttons horizontally
# buttons = HBox([button_save_data, text_display])

# # create an output widget to display the plot
# figure = Output()

# # Display buttons and output widget vertically
# display(VBox([buttons, figure]))

# # # connect events
# cid_pick = fig.canvas.mpl_connect('pick_event', on_picked_flag_point)
# cid_key_press = fig.canvas.mpl_connect('key_press_event', on_key_pressed_flag_points)

# # display the plot
# plt.show()

In [ ]:
# import matplotlib.pyplot as plt
# import numpy as np
# from ipywidgets import Button
# from ipywidgets import VBox, HBox
# from ipywidgets import Layout
# from IPython.display import display
# from ipywidgets import Output
# import matplotlib.colors as mcolors
# import polars as pl

# # Install ipympl if you haven't already
# # !pip install ipympl

# # Activate the matplotlib widget for Jupyter
# %matplotlib ipympl

# # Generate some random data
# np.random.seed(0)
# x = np.random.randn(100)
# y = np.random.randn(100)
# df = pl.DataFrame({"x": x, "y": y, "yflag": [None for i in range(100)], "color": ['blue' for i in range(100)]})


# fig, ax = plt.subplots()
# sc = ax.scatter(df["x"], df["y"], c=df["color"], s=10, picker=10, visible=True)

# # Define the recolor function
# def recolor_points(event):
#     global df
#     zoom_xlim = ax.get_xlim()
#     zoom_ylim = ax.get_ylim()
#     print(f"{zoom_xlim} -> {zoom_ylim}")
#     # for i in range(len(x)):
#     #     if zoom_xlim[0] <= x[i] <= zoom_xlim[1] and zoom_ylim[0] <= y[i] <= zoom_ylim[1]:
#     #         color_map[i] = np.random.rand(3,)
#     # colors = [color_map.get(i, None) for i in range(len(x))]
#     # sc.set_array(np.array(colors))

#     df = df.with_columns(pl.when((pl.col("x") > zoom_xlim[0]) & (pl.col("x") < zoom_xlim[1]) & (pl.col("y") > zoom_ylim[0]) & (pl.col("y") < zoom_ylim[1]))
#                          .then(pl.lit('red'))
#                          .otherwise(pl.col("color")).alias("color"))
#     sc.set_color(df["color"])
#     fig.canvas.draw()

# # Connect the key press event
# # fig.canvas.mpl_connect('key_press_event', recolor_points)

# # Create buttons
# button = Button(description="Recolor")
# button.on_click(recolor_points)
# button2 = Button(description="Unflag all")
# button3 = Button(description="Save data")

# # Arrange buttons horizontally
# buttons = HBox([button, button2, button3])

# # Create an output widget to display the plot
# output_widget = Output()

# # Display buttons and output widget vertically
# display(VBox([buttons, output_widget]))

# # Display the plot
# plt.show()
